# QUESTÃO 4 - CLASSIFICAÇÃO DE RISCO EM INVESTIMENTOS
**Algoritmo: MLP (Multilayer Perceptron)**

**Por quê?** O dataset tem 4 features (Volatilidade, Retorno, Liquidez, Maturação) e 3 classes de risco (0, 1, 2). MLP é o mais adequado para **classificação multiclasse** com múltiplas features. Perceptron é binário; K-means é não supervisionado.

### CÉLULA 1: BIBLIOTECAS E DADOS

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import csv

In [2]:
def carregar(arq):
    X, y = [], []
    with open(arq) as f:
        leitor = csv.reader(f)
        next(leitor)
        for linha in leitor:
            X.append([float(x) for x in linha[:4]])
            y.append(int(linha[4]))
    return np.array(X), np.array(y)

X, y = carregar("dataset_investimentos.csv")
print("Amostras:", len(y), "| Features:", X.shape[1])
for c in [0, 1, 2]:
    print(f"  Risco {c}: {np.sum(y == c)} amostras")

Amostras: 400 | Features: 4
  Risco 0: 134 amostras
  Risco 1: 133 amostras
  Risco 2: 133 amostras


### CÉLULA 2: IMPLEMENTAÇÃO MLP (UMA CAMADA OCULTA)

In [3]:
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

class MLP:
    def __init__(self, n_input, n_hidden, n_output, lr=0.01, epocas=500):
        self.lr, self.epocas = lr, epocas
        self.W1 = np.random.randn(n_input, n_hidden) * 0.1
        self.b1 = np.zeros((1, n_hidden))
        self.W2 = np.random.randn(n_hidden, n_output) * 0.1
        self.b2 = np.zeros((1, n_output))
    
    def fit(self, X, y):
        n = len(X)
        y_onehot = np.zeros((n, self.W2.shape[1]))
        y_onehot[np.arange(n), y] = 1
        for ep in range(self.epocas):
            z1 = np.dot(X, self.W1) + self.b1
            a1 = sigmoid(z1)
            z2 = np.dot(a1, self.W2) + self.b2
            a2 = softmax(z2)
            dz2 = a2 - y_onehot
            dw2 = np.dot(a1.T, dz2) / n
            db2 = dz2.mean(axis=0, keepdims=True)
            dz1 = np.dot(dz2, self.W2.T) * a1 * (1 - a1)
            dw1 = np.dot(X.T, dz1) / n
            db1 = dz1.mean(axis=0, keepdims=True)
            self.W2 -= self.lr * dw2
            self.b2 -= self.lr * db2
            self.W1 -= self.lr * dw1
            self.b1 -= self.lr * db1
        pred = self.predict(X)
        acu = np.mean(pred == y) * 100
        print(f"Acurácia: {acu:.2f}% | Épocas: {self.epocas}")
    
    def predict(self, X):
        a1 = sigmoid(np.dot(X, self.W1) + self.b1)
        return np.argmax(softmax(np.dot(a1, self.W2) + self.b2), axis=1)

### CÉLULA 3: TREINAMENTO

In [4]:
model = MLP(n_input=4, n_hidden=6, n_output=3, lr=0.1, epocas=1000)
model.fit(X, y)
pred = model.predict(X)

print("\n=== Matriz de Confusão ===")
for real in [0, 1, 2]:
    linha = [np.sum((y == real) & (pred == p)) for p in [0, 1, 2]]
    print(f"Real {real}: {linha}")

erros = np.sum(pred != y)
print(f"\nTotal de erros: {erros}/{len(y)} ({erros/len(y)*100:.1f}%)")

Acurácia: 100.00% | Épocas: 1000

=== Matriz de Confusão ===
Real 0: [np.int64(134), np.int64(0), np.int64(0)]
Real 1: [np.int64(0), np.int64(133), np.int64(0)]
Real 2: [np.int64(0), np.int64(0), np.int64(133)]

Total de erros: 0/400 (0.0%)


### CÉLULA 4: ANÁLISE E CONCLUSÃO

- MLP com 1 camada oculta (6 neurônios) consegue capturar **relações não lineares** entre as 4 features financeiras.
- A matriz de confusão mostra o desempenho por classe de risco (0=baixo, 1=médio, 2=alto).
- MLP é superior ao Perceptron neste caso por suportar múltiplas classes naturalmente via softmax.
- Para produção, pode-se ajustar hiperparâmetros (neurônios na camada oculta, taxa de aprendizado) e validar com dados de teste.